## PHYS 4811 Project: Using Machine learning to distinguish Leptons from Hadrons using LHCb dataset

#### Author: Fionn Doheny 

#### Student number: 21431185

The first step is to load all the necceccary modules

In [2]:
#importing neccecary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

Then we can load the dataset into pandas

In [3]:
#loading the dataset into pandas
print("Loading dataset...")
training_data = pd.read_csv("training.csv")

#checking if load was successful
print(f"Dataset successfully loaded: Total number of particle tracks: {len(training_data)}")
#print("First 5 rows of dataset:")
#print(training_data.head())

Loading dataset...
Dataset successfully loaded: Total number of particle tracks: 1200000


Since I'm going to be training the algorithm to decipher weather the track which is measured is that of a hadron or a lepton, I'm going to filter the dataset to remove all of the entries where 'Ghost' is seen.

In [4]:
#filters the dataset to remove rows where the label entry is ghost
data_filtered = training_data[training_data['Label'] != 'Ghost'].copy()


Now im going to introduce a new column which is a boolean variable of 1 if the particle is a lepton and 0 if its a hadron

In [26]:
binary_map = {
    'Electron': 1,  # 1 for leptons
    'Muon': 1,      
    'Pion': 0,  # 0 for hadrons
    'Kaon': 0,
    'Proton': 0
}

data_filtered['BinaryClass'] = data_filtered['Label'].map(binary_map)
ignore_cols = {'ID', 'Label', 'Class', 'BinaryClass'} # Ignoring all metadata colums so only numerical measurements remain
features = [col for col in data_filtered.columns if col not in ignore_cols]

#verification
print("\nBinary Class Distribution")
print(data_filtered['BinaryClass'].value_counts())
print(f"Total input features selected: {len(features)}")


Binary Class Distribution
BinaryClass
0    600000
1    400000
Name: count, dtype: int64
Total input features selected: 49


Now im going to use StandardScaler to make sure that the features with massive numerical range (momentum) don't distort the error calculation

In [27]:
# First i extract the raw numpy feature matrices (X) and the target arrays (Y)
X = data_filtered[features].values
y = data_filtered['BinaryClass'].values

# Then i split the data in to training (80%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Applying the Scaler
scaler = StandardScaler()

# Fitting and transforming the training features 
# (using fit_transform so that the algorithm learns the statistical data from the training data)
X_train_scaled = scaler.fit_transform(X_train)

# Just transforming the validation data so that there is no data leakage
X_val_scaled = scaler.transform(X_val)

# verification
print(f"training features shape: {X_train_scaled.shape}")
print(f"training features shape: {X_val_scaled.shape}")

training features shape: (800000, 49)
training features shape: (200000, 49)


Now its time to start the actual machine learning!

In [28]:
def predict_particles(X, w, b):
    """
    This will calculate the linear predictions, y_hat on the dataset
    X = a 2D array of the detector features (rows=particles, columns=measurements)
    w = 1D array of the weights
    b = scalar bias term
    """
    y_hat = np.dot(X, w) + b
    return y_hat

Now I need to define a multi Layer perceptron using PyTorch. Im going to use 3 layers starting with an output of 64 features

In [29]:
class LeptonHadronNN(nn.Module):
    def __init__(self, input_dim=49): # because i have 49 elements in my dataset
        super(LeptonHadronNN, self).__init__()

        #Layer 1 going from 49 inputs to 64 neurons
        self.fc1 = nn.Linear(in_features=input_dim, out_features=64) # Linear class type because i started with linear equations y = wx + b
        self.relu = nn.ReLU()

        #Layer 2 going from the 64 neurons to 32
        self.fc2 = nn.Linear(in_features=64, out_features=32)

        #Layer 3 going from 32 to 1 (which is the output score required for binary classification of 1 or 0 (Lepton or Hadron))
        self.fc3 = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        # This defines the forward flow so that i start with the 49 features and end up with 1
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

model = LeptonHadronNN(input_dim=49)
print(model)

LeptonHadronNN(
  (fc1): Linear(in_features=49, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
)


Now I have to write the training script for the nural network. Ill use 5 epochs: Forward pass --> compute loss --> zero gradients (so that current calculations are not contaminated with old gradient calcs) -- > backwards pass --> optimse (using adam) to finally reduce the error

In [30]:
# step 1: initiate the loss function and optimizer (since the model has already been defined in the previous cell)
criterion = nn.BCEWithLogitsLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

# step 2: convert scaled numpy arrays into PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # so the shape is (N, 1)

# Training loop over the 5 epochs:
epochs = 5
for epoch in range(epochs):
    # A: clear old gradients
    optimiser.zero_grad()

    #B: Forward pass to make the predictions
    outputs = model(X_train_tensor)

    #C: Compute loss by comparing with true data
    loss = criterion(outputs, y_train_tensor)

    #D: Backwards pass to compute gradients using autograd
    loss.backward()

    #E: Optimiser step (updating model weights)
    optimiser.step()

    # confirmation
    print(f"Echoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Echoch [1/5], Loss: 0.6839
Echoch [2/5], Loss: 0.6794
Echoch [3/5], Loss: 0.6750
Echoch [4/5], Loss: 0.6707
Echoch [5/5], Loss: 0.6663


Now its time to test the algorithm on the validation data that we split from the original dataset right near the begining! Note that im using the 'torch.no_grad()' here because I have already finalised the weights and so i dont want to continue calculating gradients and changing weights with this data.

In [33]:
# first i must make sure that the model is in evaluation mode
model.eval()

# then i have to convert the validation NumPy arrays into PyTorch Tensors
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

# then i can evaluate the data without computing the gradients
with torch.no_grad():
    # getting the raw output scores (logits) from the model
    val_logits = model(X_val_tensor)

    # passing the logits through the sigmoid to get continuous probabilities between 0 and 1
    val_probs = torch.sigmoid(val_logits)

    # converting the probabilities to binary predictions 
    val_preds = (val_probs >= 0.5).float()

    # finally i calculate the accuracy
    correct = (val_preds == y_val_tensor).sum().item()
    accuracy = correct / y_val_tensor.shape[0]

print(f"Validation Accuracy: {accuracy * 100:.2f}%")

Validation Accuracy: 88.20%


The first time i ran this it yeilded a validation accuracy of 60.20% which is pretty low. This is likely because of the fact that after only 5 epochs (in combination with a learning rate of only 0.01) the algorithm didnt have enough time to actually learn anything. I'm going to increase the number of epochs in the cell below to see if it makes a difference

In [32]:
epochs = 100  # Increased from 5 to 100

for epoch in range(epochs):
    optimiser.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimiser.step()
    
    # print every 10 epochs so it doesn't spam the screen
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/100], Loss: 0.6620
Epoch [10/100], Loss: 0.6189
Epoch [20/100], Loss: 0.5564
Epoch [30/100], Loss: 0.4906
Epoch [40/100], Loss: 0.4411
Epoch [50/100], Loss: 0.4057
Epoch [60/100], Loss: 0.3774
Epoch [70/100], Loss: 0.3535
Epoch [80/100], Loss: 0.3320
Epoch [90/100], Loss: 0.3116
Epoch [100/100], Loss: 0.2928


Again, I'm going to copy the validation cell to see if there is a difference in the accuracy now 

In [34]:
model.eval()

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

with torch.no_grad():
    val_logits = model(X_val_tensor)
    val_probs = torch.sigmoid(val_logits)
    val_preds = (val_probs >= 0.5).float()
    correct = (val_preds == y_val_tensor).sum().item()
    accuracy = correct / y_val_tensor.shape[0]

print(f"Validation Accuracy: {accuracy * 100:.2f}%")

Validation Accuracy: 88.20%


An additional way that i can try to increase the accuracy is to increase the number of layers in my perceptron from 3 to say 5. To do this Ill start with 256 features and go down from there to 128 then 64 then 32 then 1. These numbers are used because they are powers of 2 which computers find easier to work with.

In [21]:
class LeptonHadronNN(nn.Module):
    def __init__(self, input_dim=49): # because i have 49 elements in my dataset
        super(LeptonHadronNN, self).__init__()

        #Layer 1 going from 49 inputs to 256 neurons
        self.fc1 = nn.Linear(in_features=input_dim, out_features=256) # Linear class type because i started with linear equations y = wx + b
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(in_features=256, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=64)
        self.fc4 = nn.Linear(in_features=64, out_features=32)
        self.fc5 = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        # Re-defining the forward flow to add in the extra layers with more neurons
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.fc4(x)
        x = self.relu(x)
        x = self.fc5(x)
        return x

model = LeptonHadronNN(input_dim=49)


Intrestingly enough, increasing the number of layers actually significantly harmed the accuracy and the latest reading was 39.85%! After some research, I found out that this is because splitting the original 49 features into 256 actually meant that the individual element didnt have anywhere near enough information about the original feature and so when it got back down to 1 after the 5 layers, the data was not representative 